In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sber_data = pd.read_csv('data/sber_data.csv', sep=',')

Иногда данные могут содержать повторяющиеся записи — дубликаты.

**Дубликатами** называют записи, для которых значения (всех или большинства) признаков совпадают.

Дублирующаяся информация никогда не приводит ни к чему хорошему. Одинаковые записи не несут полезной информации и искажают реальную статистику. Модель несколько раз видит одно и то же наблюдение и начинает подстраиваться под него. Если дубликатов много, это может стать большой проблемой при обучении.

Способ обнаружения дубликатов зависит от того, что именно считается дубликатом. Например, за дубликаты можно посчитать записи, у которых совпадают все признаки или их часть. Если в таблице есть столбец с уникальным идентификатором (id), можно попробовать поискать дубликаты по нему: одинаковые записи могут иметь одинаковый id.

Для этого можно сравнить число уникальных значений в столбце id со всем числом строк в таблице.

In [2]:
sber_data['id'].nunique() == sber_data.shape[0]

True

Вроде бы все в порядке — каждой записи в таблице соответствует свой уникальный идентификатор. Но это еще не значит, что в таблице нет дубликатов.

Столбец id задает каждой строке свой уникальный номер, поэтому сама по себе каждая строка является уникальной. Однако содержимое других столбцов еще может повторяться.

Чтобы отследить дубликаты, можно воспользоваться методом duplicated(), который возвращает булеву маску для фильтрации. Для записей, у которых совпадают признаки, переданные методу, он возвращает True, для остальных — False.

У метода есть параметр subset — список признаков, по которым производится поиск дубликатов. По умолчанию используются все столбцы в датафрейме и ищутся полные дубликаты.

Найдем число полных дубликатов в таблице sber_data. Предварительно создадим список столбцов dupl_columns, по которым будем искать совпадения (все столбцы, не включая id, от которого мы и вовсе избавимся).

Создадим маску дубликатов с помощью метода duplicated() и произведем фильтрацию. Результат заносим в переменную sber_duplicates. Выведем число строк в результирующем датафрейме:

In [3]:
dupl_columns = list(sber_data.columns)
dupl_columns.remove('id')

mask = sber_data.duplicated(subset=dupl_columns)
sber_duplicates = sber_data[mask]
print(f'Число найденных дубликатов: {sber_duplicates.shape[0]}')

Число найденных дубликатов: 562


562 строки в таблице являются полными копиями других записей.

Теперь необходимо от них избавиться. Для этого легче всего воспользоваться методом drop_duplicates(), который удаляет повторяющиеся записи из таблицы.

Для этого создадим новую таблицу sber_dedupped, которая будет очищенной от дубликатов версией исходной таблицы.

In [4]:
sber_dedupped = sber_data.drop_duplicates(subset=dupl_columns)
print(f'Результирующее число записей: {sber_dedupped.shape[0]}')


Результирующее число записей: 29909


Отсутствием новой и полезной информации могут похвастаться не только отдельные записи, но и целые признаки.

**Неинформативными** считаются признаки, в которых большая часть строк содержит либо одинаковые значения (например, пол клиентов в мужском барбершопе), либо наоборот — признак, в котором для большинства записей значения уникальны (например, номер телефона клиента).

Пример неинформативного признака — признак адреса дома с уникальным значением для каждого дома в датасете Мельбурна.

Неинформативные признаки не играют роли при моделировании и лишь засоряют таблицу, увеличивая размерность данных. Они усиливают уже знакомое нам проклятие размерности, которое увеличивает время обучения модели и потенциально может снизить ее качество. Поэтому от таких признаков необходимо избавляться.

Чтобы считать признак неинформативным, прежде всего нужно задать какой-то порог. Например, часто используются пороги в **0.95 и 0.99**. Это означает: признак неинформативен, если в нем **95% (99%) одинаковых значений или же 95% (99%) данных полностью уникальны**.

В pandas нет методов, которые мгновенно бы выдавали список столбцов, обладающих низкой информативностью, но процедура их поиска легко реализуется вручную.

**Алгоритм поиска неинформативных признаков:**

1. Создаем пустой список low_information_cols, куда будем добавлять названия признаков, которые посчитаем неинформативными;
2. В цикле пройдемся по всем именам столбцов в таблице и для каждого будем совершать следующие действия:
    * рассчитаем top_freq — наибольшую относительную частоту — с помощью метода value_counts() с параметром normalize=True. Метод вернет долю от общих данных, которую занимает каждое уникальное значение в признаке. Отсюда нам нужен максимум с помощью метода max();
    * рассчитаем nunique_ratio — отношение числа уникальных значений в столбце к размеру всего столбца. Число уникальных значений столбца получается с помощью метода nunique(), а размер признака — с помощью метода count(). Например, для столбца id число уникальных значений 30471, оно же равно размеру таблицы. Поэтому результат отношения будет равен 1.
    * сравним каждое из полученных чисел с пороговым значением (в нашем случае 0.95) и добавим в список неинформативных признаков, если условие истинно.

In [5]:
# список неинформативных признаков
low_information_cols = []

# цикл по всем столбцам
for col in sber_data.columns:
    # наибольшая относительная частота в признаке
    top_freq = sber_data[col].value_counts(normalize=True).max()
    # доля уникальных значений от размера признака
    nunique_ratio = sber_data[col].nunique() / sber_data[col].count()
    # сравниваем наибольшую частоту с порогом
    if top_freq > 0.95:
        low_information_cols.append(col)
        print(f'{col}: {round(top_freq*100, 2)}% одинаковых значений')
    # сравниваем долю уникальных значений с порогом
    if nunique_ratio > 0.95:
        low_information_cols.append(col)
        print(f'{col}: {round(nunique_ratio*100, 2)}% уникальных значений')

id: 100.0% уникальных значений
oil_chemistry_raion: 99.03% одинаковых значений
railroad_terminal_raion: 96.27% одинаковых значений
nuclear_reactor_raion: 97.17% одинаковых значений
big_road1_1line: 97.44% одинаковых значений
mosque_count_1000: 98.08% одинаковых значений


Итого нашли шесть неинформативных признаков. Теперь их можно удалить с помощью метода drop(), передав результирующий список в его аргументы.

In [7]:
information_sber_data = sber_data.drop(low_information_cols, axis=1)
print(f'Результирующее число признаков: {information_sber_data.shape[1]}')

Результирующее число признаков: 55


Однако стоит быть внимательными и рассудительными при поиске неинформативных признаков, иначе в любом случае есть риск потерять важные данные. Лучшее решение — сначала использовать все признаки для построения базовой модели, а уже потом выбирать те, которые обладают наибольшей информативностью.

На самом деле информативность признаков определяется не только числом уникальных значений, но и их влиянием на целевой признак (тот, который мы хотим предсказать). Это называется важностью признака.

Признаки, которые обладают низкой важностью называются **нерелевантными**.

Например, если бы в данных о квартирах был признак, содержащий информацию о температуре воздуха за окном, он был бы нерелевантным.